<a href="https://colab.research.google.com/github/Shirish-12105/-Fake-Job-Posting-Detection/blob/main/fake_jod_detection1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
#  FAKE JOB POSTING DETECTION — OPTIMISED COLAB NOTEBOOK
#  Run cells top to bottom in Google Colab
# ============================================================

# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install & Imports                             ║
# ╚══════════════════════════════════════════════════════════╝
!pip install -q imbalanced-learn optuna wordcloud

import warnings, re, io, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import nltk
for pkg in ['stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score,
                              precision_recall_curve, average_precision_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
print("✅ All libraries ready!")


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — Load Dataset                                  ║
# ╚══════════════════════════════════════════════════════════╝
from google.colab import files
print("📂 Upload fake_job_postings.csv from Kaggle:")
uploaded = files.upload()
df = pd.read_csv(io.BytesIO(list(uploaded.values())[0]))

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFraud rate: {df['fraudulent'].mean()*100:.2f}%")
df.head(3)


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — EDA (fast, compact)                           ║
# ╚══════════════════════════════════════════════════════════╝
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Class balance
counts = df['fraudulent'].value_counts()
bars = axes[0].bar(['Legitimate', 'Fraudulent'], counts.values,
                   color=['#3B82F6', '#EF4444'], edgecolor='white', width=0.5)
for b in bars:
    axes[0].text(b.get_x() + b.get_width()/2, b.get_height() + 50,
                 f'{int(b.get_height())}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Class Distribution', fontsize=12)
axes[0].set_ylabel('Count')

# Missing values
miss = (df.isnull().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]
axes[1].barh(miss.index, miss.values, color='#F59E0B')
axes[1].set_title('Missing Values (%)', fontsize=12)
axes[1].set_xlabel('%')

# Feature correlation with fraud
feat_cols = ['telecommuting', 'has_company_logo', 'has_questions']
fraud_rates = df.groupby('fraudulent')[feat_cols].mean().T
fraud_rates.columns = ['Legitimate', 'Fraudulent']
fraud_rates.plot(kind='bar', ax=axes[2], color=['#3B82F6', '#EF4444'],
                 edgecolor='white', width=0.6)
axes[2].set_title('Structured Feature Rate by Class', fontsize=12)
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=15)
axes[2].legend()

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

# Word clouds
legit_text = ' '.join(df[df['fraudulent']==0]['description'].fillna('').astype(str))
fraud_text = ' '.join(df[df['fraudulent']==1]['description'].fillna('').astype(str))
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, text, cmap, title in zip(
        axes,
        [legit_text, fraud_text],
        ['Blues', 'Reds'],
        ['Legitimate Postings', 'Fraudulent Postings']):
    wc = WordCloud(width=700, height=350, background_color='white',
                   colormap=cmap, max_words=80).generate(text)
    ax.imshow(wc, interpolation='bilinear'); ax.axis('off')
    ax.set_title(f'Word Cloud — {title}', fontsize=12)
plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — Optimised Text Preprocessing                  ║
# ╚══════════════════════════════════════════════════════════╝
STOP_WORDS = set(stopwords.words('english'))
lemmatizer  = WordNetLemmatizer()

# Pre-compile regex for speed
RE_HTML    = re.compile(r'<[^>]+>')
RE_URL     = re.compile(r'http\S+|www\S+')
RE_NONALPHA = re.compile(r'[^a-z\s]')
RE_SPACE   = re.compile(r'\s+')

def clean_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    text = RE_HTML.sub(' ', text.lower())
    text = RE_URL.sub(' ', text)
    text = RE_NONALPHA.sub(' ', text)
    text = RE_SPACE.sub(' ', text).strip()
    return ' '.join(
        lemmatizer.lemmatize(w)
        for w in text.split()
        if w not in STOP_WORDS and len(w) > 2
    )

TEXT_COLS = ['title', 'company_profile', 'description', 'requirements', 'benefits']
df[TEXT_COLS] = df[TEXT_COLS].fillna('')

print("⏳ Cleaning text (vectorised apply)...")
t0 = time.time()
# Combine first, then clean once — faster than cleaning each column separately
df['combined_text'] = df[TEXT_COLS].agg(' '.join, axis=1)
df['cleaned_text']  = df['combined_text'].apply(clean_text)
print(f"✅ Done in {time.time()-t0:.1f}s")

# Feature: text length signals
df['desc_len']    = df['description'].str.split().str.len().fillna(0)
df['req_len']     = df['requirements'].str.split().str.len().fillna(0)
df['has_salary']  = df['salary_range'].notna().astype(int)
df['title_upper'] = df['title'].str.count(r'[A-Z]').fillna(0) / (df['title'].str.len()+1)

print("Sample cleaned:", df['cleaned_text'].iloc[0][:200])


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 — Feature Engineering                           ║
# ╚══════════════════════════════════════════════════════════╝
# One-hot encode categorical columns
CAT_COLS = ['employment_type', 'required_experience', 'required_education']
df_enc = pd.get_dummies(df[CAT_COLS], dummy_na=True, dtype=float)

# Binary / numeric structured features
STRUCT_COLS = ['telecommuting', 'has_company_logo', 'has_questions',
               'desc_len', 'req_len', 'has_salary', 'title_upper']
df[STRUCT_COLS] = df[STRUCT_COLS].fillna(0)

struct_all = pd.concat([df[STRUCT_COLS].reset_index(drop=True),
                        df_enc.reset_index(drop=True)], axis=1)

# TF-IDF — tuned parameters
tfidf = TfidfVectorizer(
    max_features=20_000,
    ngram_range=(1, 2),
    sublinear_tf=True,      # log(1+tf) — dampens high-frequency terms
    min_df=2,               # ignore very rare terms
    max_df=0.95,            # ignore near-universal terms
    strip_accents='unicode',
    analyzer='word'
)

X_text   = tfidf.fit_transform(df['cleaned_text'])
X_struct = csr_matrix(struct_all.values.astype(np.float32))

# Scale structured features to [0,1] to match TF-IDF range
scaler   = MaxAbsScaler()
X_struct = scaler.fit_transform(X_struct)

X = hstack([X_text, X_struct]).astype(np.float32)
y = df['fraudulent'].values

print(f"✅ Feature matrix: {X.shape}  ({X.nnz:,} non-zero entries)")
print(f"   Text features:      {X_text.shape[1]:,}")
print(f"   Structured features:{X_struct.shape[1]:,}")


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 — Stratified Split                              ║
# ╚══════════════════════════════════════════════════════════╝
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Train fraud: {y_train.mean()*100:.2f}% | Test fraud: {y_test.mean()*100:.2f}%")

# Apply SMOTE to training set only (never to test set)
smote = SMOTE(random_state=SEED, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"\nAfter SMOTE → Train: {X_train_sm.shape[0]:,} (balanced)")
print(f"  Legitimate: {(y_train_sm==0).sum():,} | Fraudulent: {(y_train_sm==1).sum():,}")


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 7 — Hyperparameter Tuning with Optuna             ║
# ╚══════════════════════════════════════════════════════════╝
print("🔍 Tuning Logistic Regression with Optuna (30 trials)...")

def lr_objective(trial):
    C   = trial.suggest_float('C', 0.01, 10.0, log=True)
    clf = LogisticRegression(C=C, max_iter=1000, solver='saga',
                              class_weight='balanced', random_state=SEED)
    cv  = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = cross_validate(clf, X_train_sm, y_train_sm, cv=cv,
                             scoring='f1', n_jobs=-1)
    return scores['test_score'].mean()

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(lr_objective, n_trials=30, show_progress_bar=True)

best_C = study.best_params['C']
print(f"\n✅ Best C = {best_C:.4f} | Best F1 = {study.best_value:.4f}")


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 8 — Train All Optimised Models                    ║
# ╚══════════════════════════════════════════════════════════╝
# CalibratedClassifierCV wraps LinearSVC to give .predict_proba()
models = {
    'Logistic Regression (tuned)': LogisticRegression(
        C=best_C, max_iter=1000, solver='saga',
        class_weight='balanced', random_state=SEED),

    'Naive Bayes': MultinomialNB(alpha=0.05),

    'Linear SVM (calibrated)': CalibratedClassifierCV(
        LinearSVC(C=0.5, class_weight='balanced',
                  max_iter=3000, random_state=SEED), cv=3),

    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        class_weight='balanced_subsample', random_state=SEED, n_jobs=-1),
}

results = {}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    elapsed = time.time() - t0

    rep = classification_report(y_test, y_pred,
                                  target_names=['Legitimate', 'Fraudulent'],
                                  output_dict=True)
    results[name] = dict(
        model     = model,
        y_pred    = y_pred,
        y_prob    = y_prob,
        accuracy  = rep['accuracy'],
        precision = rep['Fraudulent']['precision'],
        recall    = rep['Fraudulent']['recall'],
        f1        = rep['Fraudulent']['f1-score'],
        roc_auc   = roc_auc_score(y_test, y_prob),
        avg_prec  = average_precision_score(y_test, y_prob),
        train_sec = elapsed
    )
    print(f"\n{'─'*50}\n{name}  ({elapsed:.1f}s)")
    print(classification_report(y_test, y_pred,
                                  target_names=['Legitimate', 'Fraudulent']))


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 9 — Confusion Matrices                            ║
# ╚══════════════════════════════════════════════════════════╝
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (name, res) in zip(axes.flatten(), results.items()):
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, res['y_pred']),
        display_labels=['Legitimate', 'Fraudulent']
    ).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=11)
plt.suptitle('Confusion Matrices', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 10 — Model Comparison Dashboard                   ║
# ╚══════════════════════════════════════════════════════════╝
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'avg_prec']
labels  = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'Avg Precision']

comp_df = pd.DataFrame(
    {name: [res[m] for m in metrics] for name, res in results.items()},
    index=labels
).T

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
comp_df[['Recall', 'F1', 'ROC-AUC']].plot(
    kind='bar', ax=axes[0], colormap='Set2', edgecolor='white', width=0.7)
axes[0].set_title('Key Metrics — Fraud Class', fontsize=12)
axes[0].set_ylabel('Score')
axes[0].set_ylim(0.5, 1.05)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=20, ha='right')
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.3f}',
                     (p.get_x() + p.get_width()/2, p.get_height()+0.005),
                     ha='center', va='bottom', fontsize=7)

# Precision-Recall curves
for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y_test, res['y_prob'])
    axes[1].plot(rec, prec, lw=1.5,
                 label=f"{name} (AP={res['avg_prec']:.3f})")
axes[1].set_title('Precision-Recall Curves', fontsize=12)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== Full Comparison Table ===")
print(comp_df.round(4).to_string())


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 11 — Top TF-IDF Features (Logistic Regression)    ║
# ╚══════════════════════════════════════════════════════════╝
lr = results['Logistic Regression (tuned)']['model']
feat_names = np.array(tfidf.get_feature_names_out())
coef       = lr.coef_[0][:len(feat_names)]   # text features only

n = 20
top_fraud = coef.argsort()[-n:][::-1]
top_legit = coef.argsort()[:n]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].barh(feat_names[top_fraud[::-1]], coef[top_fraud[::-1]], color='#EF4444')
axes[0].set_title('Top 20 Words → Fraudulent', fontsize=12)
axes[0].set_xlabel('LR Coefficient')

axes[1].barh(feat_names[top_legit[::-1]], coef[top_legit[::-1]], color='#3B82F6')
axes[1].set_title('Top 20 Words → Legitimate', fontsize=12)
axes[1].set_xlabel('LR Coefficient')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 12 — Threshold Optimisation                       ║
# ╚══════════════════════════════════════════════════════════╝
# Default threshold (0.5) may not be optimal for imbalanced data.
# We sweep thresholds and pick the one maximising F1 on fraud class.

best_model_name = max(results, key=lambda k: results[k]['f1'])
y_prob_best     = results[best_model_name]['y_prob']

thresholds  = np.arange(0.1, 0.9, 0.01)
f1_scores   = []
rec_scores  = []
prec_scores = []

for thr in thresholds:
    y_thr = (y_prob_best >= thr).astype(int)
    rep   = classification_report(y_test, y_thr, output_dict=True,
                                   zero_division=0)
    f1_scores.append(rep['1']['f1-score'])
    rec_scores.append(rep['1']['recall'])
    prec_scores.append(rep['1']['precision'])

optimal_idx = np.argmax(f1_scores)
optimal_thr = thresholds[optimal_idx]

plt.figure(figsize=(10, 4))
plt.plot(thresholds, f1_scores,   label='F1 (Fraud)',        color='#8B5CF6', lw=2)
plt.plot(thresholds, rec_scores,  label='Recall (Fraud)',    color='#EF4444', lw=1.5, ls='--')
plt.plot(thresholds, prec_scores, label='Precision (Fraud)', color='#3B82F6', lw=1.5, ls='--')
plt.axvline(optimal_thr, color='black', ls=':', lw=1.5,
            label=f'Optimal threshold = {optimal_thr:.2f}')
plt.title(f'Threshold Sweep — {best_model_name}', fontsize=12)
plt.xlabel('Decision Threshold'); plt.ylabel('Score')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_optimisation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Optimal threshold: {optimal_thr:.2f}")
print(f"F1 at optimal:     {f1_scores[optimal_idx]:.4f}")

# Final predictions with optimal threshold
y_final = (y_prob_best >= optimal_thr).astype(int)
print("\n=== Final Report (optimal threshold) ===")
print(classification_report(y_test, y_final,
                              target_names=['Legitimate', 'Fraudulent']))


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 13 — Production Prediction System                 ║
# ╚══════════════════════════════════════════════════════════╝
class FakeJobDetector:
    """
    Production-ready fake job detector.
    Encapsulates preprocessing, vectorisation, and prediction.
    """

    def __init__(self, model, tfidf, scaler, struct_cols, threshold=0.5):
        self.model       = model
        self.tfidf       = tfidf
        self.scaler      = scaler
        self.struct_cols = struct_cols
        self.threshold   = threshold
        self._lemmatizer = WordNetLemmatizer()
        self._stopwords  = set(stopwords.words('english'))

    def _clean(self, text: str) -> str:
        if not isinstance(text, str): return ''
        text = RE_HTML.sub(' ', text.lower())
        text = RE_URL.sub(' ', text)
        text = RE_NONALPHA.sub(' ', text)
        text = RE_SPACE.sub(' ', text).strip()
        return ' '.join(
            self._lemmatizer.lemmatize(w)
            for w in text.split()
            if w not in self._stopwords and len(w) > 2
        )

    def predict(self, title='', description='', requirements='',
                company_profile='', benefits='',
                has_logo=0, telecommuting=0, has_questions=0) -> dict:

        raw_text = f"{title} {company_profile} {description} {requirements} {benefits}"
        cleaned  = self._clean(raw_text)
        X_text   = self.tfidf.transform([cleaned])

        # Build structured row — zero all, then fill known columns
        struct_row = np.zeros((1, len(self.struct_cols)), dtype=np.float32)
        col_map = {c: i for i, c in enumerate(self.struct_cols)}
        for col, val in [('telecommuting', telecommuting),
                          ('has_company_logo', has_logo),
                          ('has_questions', has_questions),
                          ('desc_len', len(description.split())),
                          ('req_len',  len(requirements.split())),
                          ('has_salary', 0)]:
            if col in col_map:
                struct_row[0, col_map[col]] = val

        X_struct  = self.scaler.transform(csr_matrix(struct_row))
        X_input   = hstack([X_text, X_struct]).astype(np.float32)
        prob      = self.model.predict_proba(X_input)[0][1]
        pred      = int(prob >= self.threshold)

        risk = 'HIGH' if prob >= 0.7 else 'MEDIUM' if prob >= self.threshold else 'LOW'
        return {
            'label':       'FRAUDULENT 🚨' if pred else 'LEGITIMATE ✅',
            'risk_level':  risk,
            'fraud_prob':  f"{prob:.2%}",
            'threshold':   f"{self.threshold:.2f}",
            'model':       best_model_name
        }


# Instantiate detector with optimal threshold
detector = FakeJobDetector(
    model       = results[best_model_name]['model'],
    tfidf       = tfidf,
    scaler      = scaler,
    struct_cols = list(struct_all.columns),
    threshold   = optimal_thr
)

# ── Test 1: Obvious scam ─────────────────────────────────
r1 = detector.predict(
    title       = "Work from home! Earn $5000/week — NO experience needed",
    description = "We are a fast-growing company. Send your personal details and a "
                  "small registration fee to get started. Guaranteed income, "
                  "unlimited earning potential. Flexible hours!",
    has_logo=0, telecommuting=1
)
print("=== Test 1 (Scam) ===")
for k, v in r1.items(): print(f"  {k:<15}: {v}")

# ── Test 2: Legitimate posting ───────────────────────────
r2 = detector.predict(
    title        = "Senior Backend Engineer — Python & Kubernetes",
    description  = "We are hiring a backend engineer to design and maintain "
                   "distributed microservices. You will lead architecture reviews "
                   "and mentor junior engineers.",
    requirements = "5+ years Python, Kubernetes, PostgreSQL, REST APIs, AWS.",
    has_logo=1, has_questions=1
)
print("\n=== Test 2 (Legitimate) ===")
for k, v in r2.items(): print(f"  {k:<15}: {v}")

# ── Test 3: Borderline case ──────────────────────────────
r3 = detector.predict(
    title       = "Marketing Assistant — remote opportunity",
    description = "Join our growing team! Commission-based role with huge potential. "
                  "No experience required. We provide training.",
    has_logo=0, telecommuting=1
)
print("\n=== Test 3 (Borderline) ===")
for k, v in r3.items(): print(f"  {k:<15}: {v}")


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 14 — Cross-Validation (final sanity check)        ║
# ╚══════════════════════════════════════════════════════════╝
print("Running 5-fold stratified cross-validation on best model...\n")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = cross_validate(
    results[best_model_name]['model'],
    X, y, cv=cv,
    scoring=['f1', 'roc_auc', 'recall'],
    n_jobs=-1
)
for metric in ['f1', 'roc_auc', 'recall']:
    scores = cv_results[f'test_{metric}']
    print(f"  {metric.upper():<10}  mean={scores.mean():.4f}  std={scores.std():.4f}  "
          f"min={scores.min():.4f}  max={scores.max():.4f}")


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 15 — Save All Outputs                             ║
# ╚══════════════════════════════════════════════════════════╝
import pickle, zipfile, os

# Save model and supporting objects
artifacts = {
    'model':       results[best_model_name]['model'],
    'tfidf':       tfidf,
    'scaler':      scaler,
    'struct_cols': list(struct_all.columns),
    'threshold':   float(optimal_thr),
    'model_name':  best_model_name
}
with open('fake_job_detector.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

# Save comparison table
comp_df.round(4).to_csv('model_comparison.csv')

# Zip everything for download
with zipfile.ZipFile('fake_job_detection_results.zip', 'w') as zf:
    for fname in ['fake_job_detector.pkl', 'model_comparison.csv',
                  'eda_overview.png', 'wordclouds.png', 'confusion_matrices.png',
                  'model_comparison.png', 'feature_importance.png',
                  'threshold_optimisation.png']:
        if os.path.exists(fname):
            zf.write(fname)

print(f"✅ All outputs saved!")
print(f"   Best model : {best_model_name}")
print(f"   Fraud F1   : {results[best_model_name]['f1']:.4f}")
print(f"   ROC-AUC    : {results[best_model_name]['roc_auc']:.4f}")
print(f"   Threshold  : {optimal_thr:.2f}")

# Download zip
from google.colab import files
files.download('fake_job_detection_results.zip')
print("📦 Download started!")
